In [69]:
%run ../pathutils.ipynb
%run ../database.ipynb
%run ../export.ipynb
%run database.ipynb
%run pathutils.ipynb

In [70]:
import yaml
from pathlib import Path

# Load the report parameters
parameter_file = Path(get_project_root_folder()) / "_data" / "year_in_the_life_of.yml"
with parameter_file.open("r", encoding="utf-8") as f:
    parameters = yaml.safe_load(f)

In [71]:
import pandas as pd
from datetime import datetime

def load_data(start_date, species, location, breeding_season_analysis):
    # Build the SQL query
    sql_file = "location_sightings_with_young.sql" if breeding_season_analysis else "location_sightings.sql"
    query = construct_query("wildlife", sql_file, {
        "START_DATE": start_date,
        "SPECIES": species,
        "LOCATION": location
    })

    # Query the data 
    try:
        df = query_data("wildlife", query)
    except ValueError:
        return None

    # Make sure the date is a date and determine the month
    df["Date"] = pd.to_datetime(df["Date"])
    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month

    return df

In [72]:
import numpy as np
from statsmodels.nonparametric.smoothers_lowess import lowess
from scipy.interpolate import PchipInterpolator


def calculate_smoothed_curve(df, value_column, lowess_frac=0.4, n_points=400):
    """
    Generate a smoothed seasonal curve from monthly data while respecting true absence.

    This function takes a monthly time series (typically 1–12 for months) and produces
    a smooth curve suitable for plotting alongside bar charts of counts or presence.

    The approach is designed specifically for ecological data, where zero values often
    represent true absence (e.g. migratory species not present in certain months),
    rather than simply low counts.

    Key steps:

    1. Identify contiguous segments of non-zero months.
       These represent periods when the species is present.

    2. For each segment, include the immediately adjacent zero months (if present)
       as boundary anchors. This ensures the curve naturally falls to zero at the
       correct ecological boundaries (e.g. arrival/departure months).

    3. Apply smoothing within each segment only:
       - Single-point segments - plotted as a flat value at that month
       - Two-point segments - linear interpolation
       - Three-point segments - shape-preserving interpolation (PCHIP)
       - Longer segments - LOWESS smoothing followed by PCHIP interpolation

       This avoids overfitting or artefacts when data is sparse.

    4. Combine all segment curves into a single smooth series.
       Months outside segments remain at zero, preserving true absence.

    5. Clip any negative values to zero.

    The result is a curve that:
    - is smooth where data exists
    - drops cleanly to zero at seasonal boundaries
    - does not interpolate across periods of absence

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing at least a 'Month' column (1–12) and the value column.
    value_column : str
        Column name containing the values to smooth (e.g. 'total_count' or 'visits').
    lowess_frac : float, optional
        Fraction of data used for LOWESS smoothing (only applied to longer segments).
    n_points : int, optional
        Number of points used to generate the smooth curve.

    Returns
    -------
    x_smooth : numpy.ndarray
        Evenly spaced month values for plotting the smooth curve.
    y_smooth : numpy.ndarray
        Smoothed values corresponding to x_smooth.
    """
    df = df.copy()

    x = df["Month"].to_numpy()
    y = df[value_column].to_numpy()

    x_smooth = np.linspace(x.min(), x.max(), n_points)
    y_smooth = np.zeros_like(x_smooth, dtype=float)

    is_nonzero = y > 0

    # Find contiguous runs of non-zero months
    segments = []
    current = []

    for i, nz in enumerate(is_nonzero):
        if nz:
            current.append(i)
        else:
            if current:
                segments.append(current)
                current = []
    if current:
        segments.append(current)

    for seg in segments:
        start = seg[0]
        end = seg[-1]

        # Include neighbouring zero month before/after if present
        seg_indices = seg.copy()
        if start > 0 and y[start - 1] == 0:
            seg_indices = [start - 1] + seg_indices
        if end < len(y) - 1 and y[end + 1] == 0:
            seg_indices = seg_indices + [end + 1]

        x_seg = x[seg_indices]
        y_seg = y[seg_indices]

        mask = (x_smooth >= x_seg.min()) & (x_smooth <= x_seg.max())

        if len(x_seg) == 1:
            month_mask = np.abs(x_smooth - x_seg[0]) < 0.5
            y_smooth[month_mask] = y_seg[0]

        elif len(x_seg) == 2:
            y_smooth[mask] = np.interp(x_smooth[mask], x_seg, y_seg)

        elif len(x_seg) == 3:
            # For short segments, interpolate directly through the points
            interpolator = PchipInterpolator(x_seg, y_seg)
            y_smooth[mask] = interpolator(x_smooth[mask])

        else:
            # LOWESS on longer segments, then shape-preserving interpolation
            smoothed = lowess(y_seg, x_seg, frac=lowess_frac, return_sorted=True)
            x_lowess = smoothed[:, 0]
            y_lowess = smoothed[:, 1]
            interpolator = PchipInterpolator(x_lowess, y_lowess)
            y_smooth[mask] = interpolator(x_smooth[mask])

    y_smooth = np.clip(y_smooth, 0, None)
    return x_smooth, y_smooth

In [73]:
import pandas as pd

def fill_missing_months(df, value_columns):
    """
    Given a dataframe and a set of column names, make sure each of those columns is represented in the data frame
    for each month, filling missing values with 0
    """
    all_months = pd.DataFrame({"Month": range(1, 13)})
    df_filled = all_months.merge(df, on="Month", how="left")

    for col in value_columns:
        df_filled[col] = df_filled[col].fillna(0)

    return df_filled

In [74]:
def longest_zero_run(arr):
    """
    Calculate the longest run of 0s in a specified array of data representing species sightings
    """
    longest = 0
    current = 0
    for v in arr:
        if v == 0:
            current += 1
            longest = max(longest, current)
        else:
            current = 0
    return longest

In [75]:
import numpy as np

def circular_month_mean(arr):
    """
    Compute the circular (seasonal) mean month for a set of monthly values.

    This function treats months as points on a circle rather than a linear scale,
    allowing correct averaging across the December–January boundary.

    For example:
    - A standard mean of [12, 1] gives 6.5 (July) - incorrect
    - A circular mean correctly gives ~1 (January)

    Method:
    1. Represent each month (1–12) as an angle on the unit circle.
       Month 1 - 0 radians, Month 12 - just under 2π.

    2. Convert each month into Cartesian coordinates:
       x = cos(angle), y = sin(angle)

    3. Weight each point by its corresponding value (e.g. sightings or presence).

    4. Compute the weighted average position (mean vector).

    5. Convert the resulting vector back into an angle.

    6. Convert that angle back into a month number (1–12).

    The result represents the "centre of activity" through the year.

    Parameters
    ----------
    arr : array-like
        Monthly values (length 12), typically counts or presence.

    Returns
    -------
    float
        The mean month (1–12, fractional), representing the seasonal centre.
    """
    months = np.arange(1, 13)
    weights = np.array(arr, dtype=float)
    if weights.sum() == 0:
        return np.nan

    angles = 2 * np.pi * (months - 1) / 12
    x = np.sum(weights * np.cos(angles))
    y = np.sum(weights * np.sin(angles))
    angle = np.arctan2(y, x)
    if angle < 0:
        angle += 2 * np.pi
    month = 1 + (angle * 12 / (2 * np.pi))
    return month

In [76]:
import numpy as np

def seasonal_width(arr, threshold_frac=0.25):
    """
    Estimate the width of a species' active season.

    This function measures how broadly a species is distributed through the year
    by counting how many months have values above a defined fraction of the peak.

    Rather than simply counting non-zero months, this focuses on the *core season*,
    ignoring very low values or occasional records.

    Method:
    1. Identify the maximum value in the series.
    2. Define a threshold as a fraction of this maximum (default = 25%).
    3. Count the number of months where the value meets or exceeds this threshold.

    The result is the number of months that make up the main period of activity.

    Interpretation:
    - Small values (e.g. 2–4 months):
        Narrow, sharply defined season (e.g. short flowering period, tight migration window)
    - Intermediate values (e.g. 4–6 months):
        Moderate season length
    - Large values (e.g. 6–12 months):
        Broad or extended season (e.g. resident species or long flowering period)

    Parameters
    ----------
    arr : array-like
        Monthly values (length 12), typically counts or presence.
    threshold_frac : float, optional
        Fraction of the maximum value used to define the "active" season.

    Returns
    -------
    int
        Number of months within the main seasonal window.
    """
    arr = np.array(arr, dtype=float)
    if arr.max() == 0:
        return 0
    threshold = arr.max() * threshold_frac
    return int(np.sum(arr >= threshold))

In [77]:
import numpy as np

def coefficient_of_variation(arr):
    """
    Compute the coefficient of variation (CV) for a monthly series.

    The coefficient of variation measures how variable a series is relative
    to its mean. It is defined as:

        CV = standard deviation / mean

    This provides a scale-independent measure of variability, allowing
    comparison between species with very different absolute counts.

    Interpretation:
    - Low CV (~0-0.3):
        Relatively stable through the year (e.g. consistent presence)
    - Moderate CV (~0.3-0.7):
        Noticeable seasonal variation
    - High CV (>0.7):
        Strongly seasonal or highly uneven distribution

    In this context:
    - High CV often indicates strong seasonality (e.g. migrants, flowering peaks)
    - Low CV suggests year-round presence or gradual variation

    Parameters
    ----------
    arr : array-like
        Monthly values (length 12), typically counts or presence.

    Returns
    -------
    float
        Coefficient of variation (standard deviation divided by mean).
        Returns NaN if the mean is zero.
    """
    arr = np.array(arr, dtype=float)
    mean = arr.mean()
    if mean == 0:
        return np.nan
    return arr.std(ddof=0) / mean

In [78]:

def extract_seasonal_features(y_counts, y_presence):
    """
    Derive a set of seasonal descriptors from monthly count and presence data.

    This function converts raw monthly time series into a structured set of
    features describing the timing, duration, and variability of a species'
    activity through the year.

    It uses both total counts (abundance) and presence (frequency of encounter)
    to capture different aspects of seasonal behaviour.

    The resulting features can be used to:
    - interpret seasonal patterns (e.g. migration, detectability, aggregation)
    - compare species
    - support rule-based classification of seasonal types

    Feature groups include:

    - Occupancy:
        Number of months with non-zero values

    - Absence structure:
        Longest run of zero months (indicating seasonal absence)

    - Timing:
        Peak month and circular mean (seasonal centre of activity)

    - Seasonal breadth:
        Width of the active season based on thresholded values

    - Variability:
        Coefficient of variation and max-to-mean ratios

    - Behavioural signal:
        Aggregation index comparing variability in counts vs presence

    Parameters
    ----------
    y_counts : array-like
        Monthly total counts (length 12)
    y_presence : array-like
        Monthly presence values (length 12)

    Returns
    -------
    dict
        Dictionary of derived seasonal features.
    """
    y_counts = np.array(y_counts, dtype=float)
    y_presence = np.array(y_presence, dtype=float)

    winter_month_idx = [10, 11, 0, 1]   # Nov, Dec, Jan, Feb
    summer_month_idx = [3, 4, 5, 6, 7]  # Apr–Aug

    features = {
        "occupied_months_counts": int(np.sum(y_counts > 0)),
        "occupied_months_presence": int(np.sum(y_presence > 0)),
        "longest_zero_run_counts": longest_zero_run(y_counts),
        "longest_zero_run_presence": longest_zero_run(y_presence),
        "peak_month_counts": int(np.argmax(y_counts) + 1),
        "peak_month_presence": int(np.argmax(y_presence) + 1),
        "centre_month_counts": circular_month_mean(y_counts),
        "centre_month_presence": circular_month_mean(y_presence),
        "seasonal_width_counts": seasonal_width(y_counts),
        "seasonal_width_presence": seasonal_width(y_presence),
        "cv_counts": coefficient_of_variation(y_counts),
        "cv_presence": coefficient_of_variation(y_presence),
        "max_to_mean_counts": float(y_counts.max() / y_counts.mean()) if y_counts.mean() else np.nan,
        "max_to_mean_presence": float(y_presence.max() / y_presence.mean()) if y_presence.mean() else np.nan,
        "aggregation_index": (
            coefficient_of_variation(y_counts) / coefficient_of_variation(y_presence)
            if coefficient_of_variation(y_presence) not in [0, np.nan] else np.nan
        ),

        # Seasonal presence features
        "annual_presence_total": float(np.sum(y_presence)),
        "annual_count_total": float(np.sum(y_counts)),
        "winter_presence_months": int(np.sum(y_presence[winter_month_idx] > 0)),
        "winter_presence_total": float(np.sum(y_presence[winter_month_idx])),
        "summer_presence_months": int(np.sum(y_presence[summer_month_idx] > 0)),
    }

    return features

In [79]:
import numpy as np

def classify_seasonal_pattern(features):
    """
    Classify a species' seasonal behaviour based on derived features.

    This is a rule-based classifier that interprets seasonal metrics such as
    occupancy, absence structure, timing, variability, aggregation, and
    winter-tail behaviour.

    The aim is not statistical optimisation, but an interpretable ecological
    categorisation aligned with observed patterns.

    ----------------------------
    How the classifier "thinks"
    ----------------------------

    The classification proceeds in a series of decision stages:

    0. Sparse / low-signal check
       - If the annual presence total is very low, the species is treated as
         too weakly represented for confident classification.
       - This helps prevent occasional or under-recorded species from being
         forced into misleading categories.

    1. Strong seasonal absence - Migrants / Visitors
       - If there is a long run of zero months (>= 4) and relatively few
         occupied months, the species is assumed to be seasonally absent for
         part of the year.
       - The timing of activity (circular mean) determines:
            • Winter-centred - "Winter visitor"
            • Summer-centred - "Summer visitor"

    2. Partial migrants
       - If there is a moderate seasonal gap (>= 2 zero months), activity is
         centred in spring/summer, and there is a genuine winter tail
         (non-zero winter presence), the species is treated as a partial migrant.
       - This distinguishes species such as Blackcap from clean summer visitors
         such as Swallow.

    3. Year-round presence - Residents
       - If the species is present in most months (>= 10) with little or no
         sustained absence, it is treated as resident.
       - Sub-classified based on behaviour:

            a. Detectability-driven resident:
               - Presence varies seasonally
               - Indicates seasonal changes in visibility or behaviour
               - (e.g. Blackbird, Robin, Wren)

            b. Aggregation-driven resident:
               - Counts vary much more than presence
               - Indicates flocking or grouping behaviour
               - (e.g. Starling, Woodpigeon)

            c. Otherwise:
               - General "Resident"

    4. Seasonal curves (typically plants or non-resident patterns)
       - If the curve is narrow (small seasonal width), classify as:
            • "Narrow seasonal curve"
       - If the curve is broad (large seasonal width), classify as:
            • "Broad seasonal curve"

    5. Fallback
       - If no rule matches clearly:
            • "Mixed / transitional"

    Parameters
    ----------
    features : dict
        Output from extract_seasonal_features()

    Returns
    -------
    tuple[str, str]
        Classification label and short explanation.
    """

    occ = features["occupied_months_presence"]
    zero_run = features["longest_zero_run_presence"]
    centre = features["centre_month_presence"]
    width = features["seasonal_width_presence"]
    cvp = features["cv_presence"]
    agg = features["aggregation_index"]

    annual_presence_total = features.get("annual_presence_total", np.nan)
    winter_presence_months = features.get("winter_presence_months", 0)
    winter_presence_total = features.get("winter_presence_total", 0.0)

    # ------------------------------------------------
    # 0. Sparse / low-signal species
    # ------------------------------------------------
    if np.isnan(annual_presence_total) or annual_presence_total < 10 or occ < 3:
        return (
            "Sparse / low-signal",
            f"Insufficient seasonal signal for confident classification "
            f"(annual presence={annual_presence_total:.1f}, occupied months={occ})."
        )

    # ------------------------------------------------
    # 1. Strong seasonal absence - migrants / visitors
    # ------------------------------------------------
    if zero_run >= 4 and occ <= 6:
        if (centre >= 10) or (centre <= 2.5):
            return (
                "Winter visitor",
                f"Strong seasonal absence with activity centred in winter "
                f"(longest zero run={zero_run}, occupied months={occ}, centre month={centre:.1f})."
            )

        if 3 <= centre <= 8:
            return (
                "Summer visitor",
                f"Strong seasonal absence with activity centred in spring/summer "
                f"(longest zero run={zero_run}, occupied months={occ}, centre month={centre:.1f})."
            )

    # ------------------------------------------------
    # 2. Partial migrants
    # ------------------------------------------------
    if zero_run >= 2 and 6 <= occ <= 10:
        if 3 <= centre <= 8:
            if winter_presence_months >= 1 and winter_presence_total > 0:
                return (
                    "Partial migrant",
                    f"Summer-centred pattern with a persistent winter tail "
                    f"(zero run={zero_run}, winter presence months={winter_presence_months}, "
                    f"winter presence total={winter_presence_total:.1f})."
                )
            else:
                return (
                    "Summer visitor",
                    f"Summer-centred pattern with seasonal absence but no meaningful winter tail "
                    f"(zero run={zero_run}, occupied months={occ})."
                )

    # ------------------------------------------------
    # 3. Residents (present most of year)
    # ------------------------------------------------
    if occ >= 10 and zero_run <= 1:

        if not np.isnan(cvp) and cvp > 0.25:
            return (
                "Detectability-driven resident",
                f"Present through most of the year, but encounter frequency varies seasonally "
                f"(occupied months={occ}, CV presence={cvp:.2f})."
            )

        if (not np.isnan(agg) and agg > 1.8) and (cvp < 0.25):
            return (
                "Aggregation-driven resident",
                f"Present through most of the year, with counts varying more strongly than presence "
                f"(aggregation index={agg:.2f}, CV presence={cvp:.2f})."
            )

        return (
            "Resident",
            f"Present through most of the year with relatively even seasonal behaviour "
            f"(occupied months={occ}, zero run={zero_run})."
        )

    # ------------------------------------------------
    # 4. Seasonal plants / general seasonal curves
    # ------------------------------------------------
    if width <= 4:
        return (
            "Narrow seasonal curve",
            f"Activity is concentrated into a relatively short seasonal window "
            f"(seasonal width={width})."
        )

    if width >= 7:
        return (
            "Broad seasonal curve",
            f"Activity is spread across a broad part of the year "
            f"(seasonal width={width})."
        )

    # ------------------------------------------------
    # 5. Fallback
    # ------------------------------------------------
    return (
        "Mixed / transitional",
        f"No single rule matched clearly; pattern appears intermediate "
        f"(occupied months={occ}, zero run={zero_run}, centre month={centre:.1f}, width={width})."
    )

In [80]:
def classify_species_from_monthly_data(monthly_totals_df, presence_df):
    """
    Given monthly totals and presence, classify the species
    """

    # Sort the monthly totals and presence by month
    y_counts = monthly_totals_df["total_count"].to_numpy()
    y_presence = presence_df["days_with_sightings"].to_numpy()

    # Extract the features and classify the species
    features = extract_seasonal_features(y_counts, y_presence)
    classification, reason = classify_seasonal_pattern(features)

    return features, classification, reason

In [81]:
def calculate_monthly_totals(df):
    """
    1. Define visits (days)
    2. Count birds per visit
    3. Count visits per month
    4. Summarise both
    """

    monthly_totals = (
        df.groupby(["Month", "Date"])
        .agg({"Count": "sum"})
        .reset_index()
        .groupby("Month")
        .agg(
            total_count=("Count", "sum"),
            visits=("Date", "nunique")
        )
    )

    # Ensure Month is a column (not index)
    monthly_totals.reset_index(inplace=True)

    # Fill in missing months
    monthly_totals = fill_missing_months(monthly_totals, ["total_count", "visits"])

    # Calculate the smoothed curve
    totals_smooth_x, totals_smooth_y = calculate_smoothed_curve(monthly_totals, "total_count")

    return monthly_totals, totals_smooth_x, totals_smooth_y

In [82]:
import numpy as np

def weighted_month_mean(y):
    """
    Linear weighted mean month (1-12), unlike circular_month_mean().
    Better suited to breeding signals, which usually do not wrap cleanly
    across the year boundary.
    """
    y = np.array(y, dtype=float)
    total = y.sum()
    if total == 0:
        return np.nan
    months = np.arange(1, 13, dtype=float)
    return float(np.sum(months * y) / total)


def fraction_after_peak(y):
    """
    Fraction of the annual signal that occurs strictly after the peak month.
    Useful for distinguishing short fledging windows from long family-association tails.
    """
    y = np.array(y, dtype=float)
    total = y.sum()
    if total == 0:
        return np.nan
    peak_idx = int(np.argmax(y))
    return float(np.sum(y[peak_idx + 1:]) / total)


def fraction_before_peak(y):
    """
    Fraction of the annual signal that occurs strictly before the peak month.
    """
    y = np.array(y, dtype=float)
    total = y.sum()
    if total == 0:
        return np.nan
    peak_idx = int(np.argmax(y))
    return float(np.sum(y[:peak_idx]) / total)


def peak_fraction(y):
    """
    Fraction of the annual signal concentrated in the peak month.
    High values indicate a compact, sharply concentrated breeding signal.
    """
    y = np.array(y, dtype=float)
    total = y.sum()
    if total == 0:
        return np.nan
    return float(np.max(y) / total)


def months_after_peak_above_fraction(y, fraction=0.25):
    """
    Number of months after the peak month that remain at or above a given
    fraction of the peak value. A good proxy for post-peak persistence.
    """
    y = np.array(y, dtype=float)
    peak = y.max()
    if peak <= 0:
        return 0

    peak_idx = int(np.argmax(y))
    threshold = peak * fraction

    return int(np.sum(y[peak_idx + 1:] >= threshold))


def active_month_span(y):
    """
    Span from first to last non-zero month, inclusive.
    Unlike seasonal_width(), this is a simple calendar span and does not assume circularity.
    """
    y = np.array(y, dtype=float)
    active = np.where(y > 0)[0]
    if len(active) == 0:
        return 0
    return int(active[-1] - active[0] + 1)


def extract_breeding_features(y_counts, y_presence):
    """
    Derive breeding-specific descriptors from monthly count and presence data.

    The main signal of interest is presence-with-young, treated as a proxy for
    observed breeding activity rather than breeding in the strict biological sense.

    Presence is treated as the primary timing signal.
    Counts are retained as a secondary measure of intensity / repeated family-group detection.
    """
    y_counts = np.array(y_counts, dtype=float)
    y_presence = np.array(y_presence, dtype=float)

    features = {
        # Basic occupancy / signal strength
        "occupied_months_counts": int(np.sum(y_counts > 0)),
        "occupied_months_presence": int(np.sum(y_presence > 0)),
        "annual_count_total": float(np.sum(y_counts)),
        "annual_presence_total": float(np.sum(y_presence)),

        # Timing
        "peak_month_counts": int(np.argmax(y_counts) + 1) if y_counts.sum() > 0 else np.nan,
        "peak_month_presence": int(np.argmax(y_presence) + 1) if y_presence.sum() > 0 else np.nan,
        "centre_month_counts": weighted_month_mean(y_counts),
        "centre_month_presence": weighted_month_mean(y_presence),

        # Breadth / confinement
        "seasonal_width_counts": seasonal_width(y_counts),
        "seasonal_width_presence": seasonal_width(y_presence),
        "active_span_counts": active_month_span(y_counts),
        "active_span_presence": active_month_span(y_presence),
        "longest_zero_run_counts": longest_zero_run(y_counts),
        "longest_zero_run_presence": longest_zero_run(y_presence),

        # Shape / concentration
        "cv_counts": coefficient_of_variation(y_counts),
        "cv_presence": coefficient_of_variation(y_presence),
        "max_to_mean_counts": float(y_counts.max() / y_counts.mean()) if y_counts.mean() else np.nan,
        "max_to_mean_presence": float(y_presence.max() / y_presence.mean()) if y_presence.mean() else np.nan,
        "peak_fraction_counts": peak_fraction(y_counts),
        "peak_fraction_presence": peak_fraction(y_presence),

        # Tail structure
        "post_peak_fraction_counts": fraction_after_peak(y_counts),
        "post_peak_fraction_presence": fraction_after_peak(y_presence),
        "pre_peak_fraction_counts": fraction_before_peak(y_counts),
        "pre_peak_fraction_presence": fraction_before_peak(y_presence),
        "post_peak_months_counts_25": months_after_peak_above_fraction(y_counts, fraction=0.25),
        "post_peak_months_presence_25": months_after_peak_above_fraction(y_presence, fraction=0.25),

        # Count-vs-presence comparison
        "count_presence_width_ratio": (
            float(seasonal_width(y_counts) / seasonal_width(y_presence))
            if seasonal_width(y_presence) not in [0, np.nan] else np.nan
        ),
        "count_presence_total_ratio": (
            float(np.sum(y_counts) / np.sum(y_presence))
            if np.sum(y_presence) > 0 else np.nan
        ),
    }

    return features

In [83]:
def classify_breeding_pattern(features):
    """
    Rule-based classifier for observed breeding activity.

    This classifier is intended to describe the shape of the dependent-young signal,
    not breeding biology in the strictest sense.

    It uses presence as the primary timing signal and counts as secondary support.
    """

    occ = features["occupied_months_presence"]
    total = features["annual_presence_total"]
    width = features["seasonal_width_presence"]
    span = features["active_span_presence"]
    peak_month = features["peak_month_presence"]
    peak_fraction_ = features["peak_fraction_presence"]
    post_peak_fraction = features["post_peak_fraction_presence"]
    post_peak_months = features["post_peak_months_presence_25"]
    # TODO: Remove this
    # cv_presence = features["cv_presence"]

    # 0. Too little signal to classify confidently
    if total < 3 or occ <= 1:
        return (
            "No breeding signal",
            "Too few records with dependent young to define a meaningful seasonal pattern."
        )

    # 1. Short, concentrated breeding signal
    # Typical of small passerines with a compact fledging window.
    if (
        occ <= 3
        and width <= 3.0
        and span <= 4
        and peak_fraction_ >= 0.45
        and post_peak_months <= 1
        and post_peak_fraction <= 0.35
    ):
        return (
            "Brief breeding window",
            f"Observed breeding activity is tightly concentrated around month {int(peak_month)}, "
            "with a short post-peak tail and little prolonged family association."
        )

    # 2. Long post-peak tail / prolonged association
    # Typical of swans, ducks, geese, or species where young remain with adults for months.
    if (
        occ >= 5
        and width >= 4.5
        and span >= 5
        and post_peak_months >= 3
        and post_peak_fraction >= 0.45
    ):
        return (
            "Extended family association",
            f"Observed breeding activity peaks around month {int(peak_month)} but remains detectable for "
            "several months afterwards, consistent with prolonged parental association."
        )

    # 3. Recognisable but not extremely narrow or prolonged
    if (
        occ >= 3
        and width >= 2.5
        and width <= 5.5
        and span <= 6
        and post_peak_fraction < 0.50
    ):
        return (
            "Moderate breeding window",
            f"Observed breeding activity forms a clear seasonal window centred around month {int(peak_month)}, "
            "broader than a brief fledging pulse but without a strongly extended tail."
        )

    # 4. Fallback
    return (
        "Diffuse / ambiguous breeding signal",
        "Observed breeding activity is present but does not resolve cleanly into a compact window "
        "or a strongly extended family-association pattern."
    )

In [84]:
def classify_breeding_from_monthly_data(monthly_totals_df, presence_df):
    """
    Given breeding monthly totals and breeding presence,
    derive breeding features and classify the breeding signal.
    """

    y_counts = monthly_totals_df["total_count"].to_numpy()
    y_presence = presence_df["days_with_sightings"].to_numpy()

    features = extract_breeding_features(y_counts, y_presence)
    classification, reason = classify_breeding_pattern(features)

    return features, classification, reason

In [85]:
from scipy.signal import find_peaks
import numpy as np

def active_month_range(y):
    """
    Return the first and last active months (1-12) in a non-circular sense.
    Useful for butterfly flight periods, which are usually best read as a
    calendar-bound seasonal window rather than a wrap-around annual signal.
    """
    y = np.array(y, dtype=float)
    active = np.where(y > 0)[0]
    if len(active) == 0:
        return np.nan, np.nan
    return int(active[0] + 1), int(active[-1] + 1)


def longest_internal_zero_run(y):
    """
    Longest run of zero months occurring *within* the active season, i.e.
    between the first and last active months.
    """
    y = np.array(y, dtype=float)
    active = np.where(y > 0)[0]
    if len(active) <= 1:
        return 0

    window = y[active[0]:active[-1] + 1]
    return longest_zero_run(window)


def count_significant_peaks(y, prominence_frac=0.20, min_distance_months=1.5):
    """
    Count ecologically meaningful peaks in a smoothed monthly curve.

    Parameters
    ----------
    y : array-like
        Smoothed y-values.
    prominence_frac : float
        Minimum prominence as a fraction of the global peak.
    min_distance_months : float
        Minimum separation between peaks in month units.

    Returns
    -------
    tuple
        (n_peaks, peak_indices, peak_heights)
    """
    y = np.array(y, dtype=float)
    if len(y) == 0 or np.max(y) <= 0:
        return 0, np.array([], dtype=int), np.array([], dtype=float)

    prominence = np.max(y) * prominence_frac
    # 400 smoothed points across ~11 months => ~36 points per month
    distance = max(1, int((len(y) / 11.0) * min_distance_months))

    peak_idx, props = find_peaks(y, prominence=prominence, distance=distance)

    heights = y[peak_idx]
    keep = heights >= (np.max(y) * 0.25)

    peak_idx = peak_idx[keep]
    heights = heights[keep]

    return int(len(peak_idx)), peak_idx, heights


def peak_month_list_from_smoothed_curve(x, y, prominence_frac=0.20, min_distance_months=1.5):
    """
    Convenience wrapper returning ecologically meaningful peak months
    from a smoothed curve.
    """
    n_peaks, peak_idx, heights = count_significant_peaks(
        y,
        prominence_frac=prominence_frac,
        min_distance_months=min_distance_months
    )

    peak_months = [int(np.clip(np.round(x[idx]), 1, 12)) for idx in peak_idx]
    return n_peaks, peak_idx, heights, peak_months


def peak_separation_metrics(y, peak_months, trough_frac=0.35):
    """
    Assess how strongly multiple peaks are separated using the unsmoothed
    monthly series.

    Parameters
    ----------
    y : array-like
        Monthly values (typically presence).
    peak_months : list[int]
        Peak months (1-12).
    trough_frac : float
        A trough is treated as 'clear' if the minimum between peaks is
        <= trough_frac * lower_peak_height.

    Returns
    -------
    dict
        {
            "has_clear_separation": bool,
            "has_zero_gap": bool,
            "min_trough_ratio": float or np.nan,
            "n_clear_gaps": int
        }
    """
    y = np.array(y, dtype=float)

    if len(peak_months) < 2:
        return {
            "has_clear_separation": False,
            "has_zero_gap": False,
            "min_trough_ratio": np.nan,
            "n_clear_gaps": 0
        }

    peak_months = sorted(set(int(m) for m in peak_months if 1 <= int(m) <= 12))

    trough_ratios = []
    clear_gaps = 0
    zero_gap = False

    for m1, m2 in zip(peak_months[:-1], peak_months[1:]):
        i1 = m1 - 1
        i2 = m2 - 1

        peak1 = y[i1]
        peak2 = y[i2]
        lower_peak = min(peak1, peak2)

        if i2 - i1 <= 1:
            trough = min(peak1, peak2)
        else:
            between = y[i1 + 1:i2]
            trough = np.min(between) if len(between) > 0 else min(peak1, peak2)

        if lower_peak > 0:
            trough_ratio = float(trough / lower_peak)
        else:
            trough_ratio = np.nan

        trough_ratios.append(trough_ratio)

        if trough == 0:
            zero_gap = True

        if np.isfinite(trough_ratio) and trough_ratio <= trough_frac:
            clear_gaps += 1

    min_trough_ratio = np.nanmin(trough_ratios) if len(trough_ratios) > 0 else np.nan

    return {
        "has_clear_separation": clear_gaps > 0,
        "has_zero_gap": zero_gap,
        "min_trough_ratio": min_trough_ratio,
        "n_clear_gaps": clear_gaps
    }


def extract_flight_period_features(y_counts, y_presence):
    """
    Derive butterfly flight-period descriptors from monthly totals and presence.

    Presence is treated as the primary phenology signal; counts are retained as
    a secondary measure of relative intensity.
    """
    y_counts = np.array(y_counts, dtype=float)
    y_presence = np.array(y_presence, dtype=float)

    presence_df = pd.DataFrame({
        "Month": np.arange(1, 13),
        "days_with_sightings": y_presence
    })
    counts_df = pd.DataFrame({
        "Month": np.arange(1, 13),
        "total_count": y_counts
    })

    px, py = calculate_smoothed_curve(presence_df, "days_with_sightings")
    tx, ty = calculate_smoothed_curve(counts_df, "total_count")

    peak_count_presence, peak_idx_presence, peak_heights_presence, peak_months_presence = peak_month_list_from_smoothed_curve(
        px, py
    )
    peak_count_counts, peak_idx_counts, peak_heights_counts, peak_months_counts = peak_month_list_from_smoothed_curve(
        tx, ty
    )

    separation = peak_separation_metrics(y_presence, peak_months_presence, trough_frac=0.35)

    start_month, end_month = active_month_range(y_presence)

    features = {
        # Basic signal strength
        "occupied_months_counts": int(np.sum(y_counts > 0)),
        "occupied_months_presence": int(np.sum(y_presence > 0)),
        "annual_count_total": float(np.sum(y_counts)),
        "annual_presence_total": float(np.sum(y_presence)),

        # Timing / extent
        "start_month_presence": start_month,
        "end_month_presence": end_month,
        "peak_month_counts": int(np.argmax(y_counts) + 1) if np.sum(y_counts) > 0 else np.nan,
        "peak_month_presence": int(np.argmax(y_presence) + 1) if np.sum(y_presence) > 0 else np.nan,
        "centre_month_counts": weighted_month_mean(y_counts),
        "centre_month_presence": weighted_month_mean(y_presence),

        # Breadth / gaps
        "seasonal_width_counts": seasonal_width(y_counts),
        "seasonal_width_presence": seasonal_width(y_presence),
        "active_span_counts": active_month_span(y_counts),
        "active_span_presence": active_month_span(y_presence),
        "longest_zero_run_counts": longest_zero_run(y_counts),
        "longest_zero_run_presence": longest_zero_run(y_presence),
        "longest_internal_zero_run_counts": longest_internal_zero_run(y_counts),
        "longest_internal_zero_run_presence": longest_internal_zero_run(y_presence),

        # Shape / concentration
        "cv_counts": coefficient_of_variation(y_counts),
        "cv_presence": coefficient_of_variation(y_presence),
        "peak_fraction_counts": peak_fraction(y_counts),
        "peak_fraction_presence": peak_fraction(y_presence),
        "post_peak_fraction_counts": fraction_after_peak(y_counts),
        "post_peak_fraction_presence": fraction_after_peak(y_presence),

        # Peak structure
        "peak_count_counts": peak_count_counts,
        "peak_count_presence": peak_count_presence,
        "peak_months_counts": peak_months_counts,
        "peak_months_presence": peak_months_presence,
        "latest_peak_month_presence": max(peak_months_presence) if peak_months_presence else np.nan,

        # Peak separation / brood separation
        "has_clear_peak_separation_presence": separation["has_clear_separation"],
        "has_zero_gap_between_peaks_presence": separation["has_zero_gap"],
        "min_trough_ratio_presence": separation["min_trough_ratio"],
        "clear_gap_count_presence": separation["n_clear_gaps"],

        # Count vs presence
        "count_presence_width_ratio": (
            float(seasonal_width(y_counts) / seasonal_width(y_presence))
            if seasonal_width(y_presence) not in [0, np.nan] else np.nan
        ),
        "count_presence_total_ratio": (
            float(np.sum(y_counts) / np.sum(y_presence))
            if np.sum(y_presence) > 0 else np.nan
        ),
    }

    return features


def classify_flight_period(features):
    """
    Rule-based classifier for butterfly flight-period structure.

    The aim is to describe the *shape* of annual adult activity as observed in
    the field, rather than to infer the full life-history of the species.
    """

    total = features["annual_presence_total"]
    occ = features["occupied_months_presence"]
    width = features["seasonal_width_presence"]
    span = features["active_span_presence"]
    peak_count_ = features["peak_count_presence"]
    peak_month = features["peak_month_presence"]
    latest_peak = features["latest_peak_month_presence"]
    peak_fraction_ = features["peak_fraction_presence"]

    peak_months = features["peak_months_presence"]
    has_clear_sep = features["has_clear_peak_separation_presence"]
    has_zero_gap = features["has_zero_gap_between_peaks_presence"]
    min_trough_ratio = features["min_trough_ratio_presence"]

    # 0. Too little signal
    if total < 3 or occ <= 1:
        return (
            "Sparse / insufficient signal",
            "Too few butterfly records to define a meaningful flight-period pattern."
        )

    # 1. Single-peak patterns
    if peak_count_ <= 1:
        if width <= 3 and span <= 4:
            return (
                "Single brood (spring)",
                f"Flight period is concentrated into a short seasonal window centred around month {int(peak_month)}."
            )

        return (
            "Single brood (extended)",
            f"Flight period forms one main peak around month {int(peak_month)}, but with a broader seasonal spread."
        )

    # 2. Two-peak patterns: structure first, not span
    if peak_count_ == 2:
        first_peak = min(peak_months) if peak_months else np.nan
        last_peak = max(peak_months) if peak_months else np.nan

        if has_clear_sep:
            # classic overwintering-adult style:
            # early peak + later peak with real separation
            if first_peak <= 4 and last_peak >= 6:
                return (
                    "Bimodal (overwintering adult)",
                    f"Two distinct flight peaks are separated by a marked dip "
                    f"(minimum trough ratio={min_trough_ratio:.2f}), with activity spanning early and later season."
                )

            return (
                "Bimodal (separated broods)",
                f"Two distinct flight peaks are present and separated by a marked dip "
                f"(minimum trough ratio={min_trough_ratio:.2f}), consistent with two broods."
            )

        return (
            "Multi-brood (overlapping)",
            "Two flight peaks are present, but without a strong separating dip; "
            "this suggests overlapping broods or a continuous brood sequence."
        )

    # 3. Three or more peaks
    if peak_count_ >= 3:
        if has_clear_sep:
            return (
                "Multi-brood",
                f"Multiple flight peaks are present, with at least one meaningful dip between broods "
                f"(minimum trough ratio={min_trough_ratio:.2f})."
            )
        return (
            "Multi-brood (overlapping)",
            "Multiple peaks are present across the season, but without strong separation between them."
        )

    # 4. Only after peak-based routes: extended / migratory
    if span >= 7 and width >= 4 and peak_fraction_ <= 0.35:
        return (
            "Extended / migratory",
            f"Flight activity is spread broadly through the year (span={span}, width={width}) "
            f"with weak overall concentration and no strongly separated brood structure."
        )

    # 5. Fallback
    return (
        "Mixed / transitional",
        "Flight pattern is present and structured, but does not resolve cleanly into a single, brood-structured, or extended type."
    )


def classify_flight_period_from_monthly_data(monthly_totals_df, presence_df):
    """
    Given monthly totals and presence, derive butterfly flight-period features
    and classify the flight pattern.
    """
    y_counts = monthly_totals_df["total_count"].to_numpy()
    y_presence = presence_df["days_with_sightings"].to_numpy()

    features = extract_flight_period_features(y_counts, y_presence)
    classification, reason = classify_flight_period(features)

    return features, classification, reason

In [86]:
def extract_flowering_features(y_counts, y_presence):
    """
    Derive flowering-period descriptors from monthly totals and presence.

    This is intended to describe the *observed flowering signal* in the notebook,
    using presence (days/month with flowering records) as the main phenology
    signal and counts as a secondary measure of intensity.

    Unlike the bird seasonality classifier, flowering is treated as a
    calendar-bound seasonal window rather than a circular annual pattern.
    """
    y_counts = np.array(y_counts, dtype=float)
    y_presence = np.array(y_presence, dtype=float)

    presence_df = pd.DataFrame({
        "Month": np.arange(1, 13),
        "days_with_sightings": y_presence
    })
    counts_df = pd.DataFrame({
        "Month": np.arange(1, 13),
        "total_count": y_counts
    })

    px, py = calculate_smoothed_curve(presence_df, "days_with_sightings")
    tx, ty = calculate_smoothed_curve(counts_df, "total_count")

    peak_count_presence, peak_idx_presence, peak_heights_presence, peak_months_presence = peak_month_list_from_smoothed_curve(
        px, py
    )
    peak_count_counts, peak_idx_counts, peak_heights_counts, peak_months_counts = peak_month_list_from_smoothed_curve(
        tx, ty
    )

    separation = peak_separation_metrics(y_presence, peak_months_presence, trough_frac=0.40)

    start_month, end_month = active_month_range(y_presence)

    features = {
        # Basic signal strength
        "occupied_months_counts": int(np.sum(y_counts > 0)),
        "occupied_months_presence": int(np.sum(y_presence > 0)),
        "annual_count_total": float(np.sum(y_counts)),
        "annual_presence_total": float(np.sum(y_presence)),

        # Timing / extent
        "start_month_presence": start_month,
        "end_month_presence": end_month,
        "peak_month_counts": int(np.argmax(y_counts) + 1) if np.sum(y_counts) > 0 else np.nan,
        "peak_month_presence": int(np.argmax(y_presence) + 1) if np.sum(y_presence) > 0 else np.nan,
        "centre_month_counts": weighted_month_mean(y_counts),
        "centre_month_presence": weighted_month_mean(y_presence),

        # Breadth / gaps
        "seasonal_width_counts": seasonal_width(y_counts),
        "seasonal_width_presence": seasonal_width(y_presence),
        "active_span_counts": active_month_span(y_counts),
        "active_span_presence": active_month_span(y_presence),
        "longest_zero_run_counts": longest_zero_run(y_counts),
        "longest_zero_run_presence": longest_zero_run(y_presence),
        "longest_internal_zero_run_counts": longest_internal_zero_run(y_counts),
        "longest_internal_zero_run_presence": longest_internal_zero_run(y_presence),

        # Shape / concentration
        "cv_counts": coefficient_of_variation(y_counts),
        "cv_presence": coefficient_of_variation(y_presence),
        "peak_fraction_counts": peak_fraction(y_counts),
        "peak_fraction_presence": peak_fraction(y_presence),
        "post_peak_fraction_counts": fraction_after_peak(y_counts),
        "post_peak_fraction_presence": fraction_after_peak(y_presence),

        # Peak structure
        "peak_count_counts": peak_count_counts,
        "peak_count_presence": peak_count_presence,
        "peak_months_counts": peak_months_counts,
        "peak_months_presence": peak_months_presence,
        "latest_peak_month_presence": max(peak_months_presence) if peak_months_presence else np.nan,

        # Peak separation / recurrence
        "has_clear_peak_separation_presence": separation["has_clear_separation"],
        "has_zero_gap_between_peaks_presence": separation["has_zero_gap"],
        "min_trough_ratio_presence": separation["min_trough_ratio"],
        "clear_gap_count_presence": separation["n_clear_gaps"],

        # Count vs presence
        "count_presence_width_ratio": (
            float(seasonal_width(y_counts) / seasonal_width(y_presence))
            if seasonal_width(y_presence) not in [0, np.nan] else np.nan
        ),
        "count_presence_total_ratio": (
            float(np.sum(y_counts) / np.sum(y_presence))
            if np.sum(y_presence) > 0 else np.nan
        ),
    }

    return features


def classify_flowering_period(features):
    """
    Rule-based classifier for observed flowering-period shape.

    The aim is not to infer absolute botanical flowering biology, but to
    describe how the flowering signal behaves in the user's local records.
    """
    total = features["annual_presence_total"]
    occ = features["occupied_months_presence"]
    width = features["seasonal_width_presence"]
    span = features["active_span_presence"]
    peak_count_ = features["peak_count_presence"]
    peak_month = features["peak_month_presence"]
    start_month = features["start_month_presence"]
    end_month = features["end_month_presence"]
    peak_fraction_ = features["peak_fraction_presence"]

    peak_months = features["peak_months_presence"]
    has_clear_sep = features["has_clear_peak_separation_presence"]
    min_trough_ratio = features["min_trough_ratio_presence"]
    longest_zero_run = features["longest_zero_run_presence"]
    internal_zero_run = features["longest_internal_zero_run_presence"]

    if total < 3 or occ <= 1:
        return (
            "Sparse / insufficient signal",
            "Too few flowering records to define a meaningful flowering-period pattern."
        )

    # -------------------------------------------------------------------------
    # Near-continuous flowering:
    # present through most of the year, even if one season is much stronger.
    # This catches species like Red Dead-Nettle that have a strong spring peak
    # but continue at lower levels for much of the rest of the year.
    # -------------------------------------------------------------------------
    if occ >= 9 and span >= 10 and longest_zero_run <= 1:
        if peak_month <= 5:
            return (
                "Near-continuous flowering (spring peak)",
                f"Flowering is recorded through most of the year, with only brief or no gaps, "
                f"but shows a pronounced spring maximum around month {int(peak_month)}."
            )
        elif 6 <= peak_month <= 8:
            return (
                "Near-continuous flowering (summer peak)",
                f"Flowering is recorded through most of the year, with only brief or no gaps, "
                f"but shows a pronounced summer maximum around month {int(peak_month)}."
            )
        elif peak_month >= 9:
            return (
                "Near-continuous flowering (late peak)",
                f"Flowering is recorded through most of the year, with only brief or no gaps, "
                f"but shows a stronger late-season maximum around month {int(peak_month)}."
            )
        else:
            return (
                "Near-continuous flowering",
                "Flowering is recorded through most of the year, with only brief or no gaps."
            )

    # Flatter all-year flowering signal without one dominant seasonal maximum
    if span >= 8 and occ >= 7 and peak_fraction_ <= 0.30:
        return (
            "Near-continuous flowering",
            f"Flowering is recorded across much of the year (months {int(start_month)}-{int(end_month)}), "
            f"with no single sharply dominant peak."
        )

    if peak_count_ >= 2:
        if has_clear_sep:
            return (
                "Bimodal / recurrent flowering",
                f"Two or more flowering peaks are present with a clear dip between them "
                f"(minimum trough ratio={min_trough_ratio:.2f})."
            )

        if span >= 6:
            return (
                "Extended / recurrent flowering",
                f"Flowering shows more than one seasonal pulse across an extended period "
                f"(months {int(start_month)}-{int(end_month)}), but without sharply separated waves."
            )

    if peak_month <= 3 and end_month <= 5 and span <= 4:
        return (
            "Early spring ephemeral",
            f"Flowering is concentrated early in the year, peaking around month {int(peak_month)} "
            f"and ending by month {int(end_month)}."
        )

    if peak_month <= 5 and end_month <= 7 and span <= 5:
        return (
            "Spring pulse",
            f"Flowering is mainly spring-centred, with a main peak around month {int(peak_month)} "
            f"and a relatively compact seasonal window."
        )

    if peak_month <= 6 and span >= 5:
        return (
            "Extended spring-summer flowering",
            f"Flowering begins in spring and continues well into summer "
            f"(months {int(start_month)}-{int(end_month)}), centred around month {int(peak_month)}."
        )

    if 6 <= peak_month <= 8:
        return (
            "Summer peak",
            f"Flowering is centred on summer, with the strongest activity around month {int(peak_month)}."
        )

    if peak_month >= 9:
        return (
            "Late-season flowering",
            f"Flowering peaks late in the year, with the main concentration around month {int(peak_month)}."
        )

    return (
        "Mixed / transitional flowering",
        f"Flowering is seasonal and structured, but does not resolve cleanly into an early, spring, summer, or recurrent type "
        f"(months {int(start_month)}-{int(end_month)})."
    )


def classify_flowering_from_monthly_data(monthly_totals_df, presence_df):
    """
    Given monthly totals and presence, derive flowering-period features
    and classify the flowering pattern.
    """
    y_counts = monthly_totals_df["total_count"].to_numpy()
    y_presence = presence_df["days_with_sightings"].to_numpy()

    features = extract_flowering_features(y_counts, y_presence)
    classification, reason = classify_flowering_period(features)

    return features, classification, reason

In [87]:
import matplotlib.pyplot as plt

def chart_monthly_totals(species, filestem, monthly_totals, totals_smooth_x, totals_smooth_y):
    plt.figure(figsize=(10, 5))

    # Draw the barchart
    plt.bar(monthly_totals["Month"], monthly_totals["total_count"], color="lightblue")

    # Smoothed line
    plt.plot(totals_smooth_x, totals_smooth_y, color="red")

    # Add labels
    plt.xlabel("Month")
    plt.ylabel("Sightings")
    plt.title(f"Monthly Sightings ({species} at {parameters["location"]})")

    # Use the shortened form of the month as the X-label
    plt.xticks(
        ticks=range(1, 13),
        labels=["Jan","Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"]
    )

    plt.tight_layout()

    # Export the chart
    export_chart(export_folder_path, f"{filestem}_totals", "png")

    plt.close()

In [88]:
def calculate_presence(df):
  """
  1. Group by date essentially defines the unit of observation (day)
  2. Size counts how many rows exist per month
  3. Reset index makes the number seen per date the index
  4. Grouping by month again aggregates
  5. The final aggregation counts how many days had >= 1 sighting
  """

  presence = (
      df.groupby(['Month', 'Date'])
        .size()
        .reset_index(name='seen')
        .groupby('Month')
        .agg(days_with_sightings=('Date', 'nunique'))
  )

  # Ensure Month is a column (not index)
  presence.reset_index(inplace=True)

  # Fill in missing months
  presence = fill_missing_months(presence, ["days_with_sightings"])

  # Calculate the smoothed curve
  presence_smooth_x, presence_smooth_y = calculate_smoothed_curve(presence, "days_with_sightings")

  return presence, presence_smooth_x, presence_smooth_y

In [89]:
import matplotlib.pyplot as plt

def chart_presence(species, filestem, presence, presence_smooth_x, presence_smooth_y):
    plt.figure(figsize=(10, 5))

    # Draw the barchart
    plt.bar(presence["Month"], presence["days_with_sightings"], color="lightblue")

    # Smoothed line
    plt.plot(presence_smooth_x, presence_smooth_y, color="red")

    # Add labels
    plt.xlabel("Month")
    plt.ylabel("Presence")
    plt.title(f"Presence ({species} at {parameters["location"]})")

    # Use the shortened form of the month as the X-label
    plt.xticks(
        ticks=range(1, 13),
        labels=["Jan","Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"]
    )

    plt.tight_layout()

    # Export the chart
    export_chart(export_folder_path, f"{filestem}_presence", "png")
    
    plt.close()

In [90]:
import re
import unicodedata

def make_safe_slug(value: str) -> str:
    """
    Convert a string into a stable ASCII-ish slug suitable for filenames.

    Rules:
    - lowercase
    - strip leading/trailing whitespace
    - convert '&' to 'and'
    - remove apostrophes
    - replace non-alphanumeric runs with underscores
    - collapse repeated underscores
    - strip leading/trailing underscores
    """
    if value is None:
        return ""

    value = str(value).strip().lower()
    value = value.replace("&", " and ")
    value = value.replace("'", "")
    value = value.replace("’", "")

    # Normalize accents/diacritics: é -> e, ü -> u, etc.
    value = unicodedata.normalize("NFKD", value)
    value = value.encode("ascii", "ignore").decode("ascii")

    # Replace any run of non-alphanumeric chars with underscore
    value = re.sub(r"[^a-z0-9]+", "_", value)

    # Collapse multiple underscores and trim
    value = re.sub(r"_+", "_", value).strip("_")

    return value

In [91]:

import pandas as pd

classification_summary = pd.DataFrame(columns=["Species", "Classification", "Reason"])
breeding_summary = pd.DataFrame(columns=["Species", "Breeding Classification", "Reason"])
flight_period_summary = pd.DataFrame(columns=["Species", "Flight Period Classification", "Reason"])
flowering_summary = pd.DataFrame(columns=["Species", "Flowering Classification", "Reason"])

# Calculate a location name without illegal path characters in it
safe_location = make_safe_slug(parameters["location"])

# Calculate the export folder path
export_folder_path = get_year_in_the_life_folder_path()

# Iterate over the species of interest
for entry in parameters["species"]:
    species = entry["name"]
    classify_seasonality = entry.get("classify_seasonality", 0) == 1
    classify_breeding = entry.get("classify_breeding", 0) == 1
    do_classify_flight_period = entry.get("classify_flight_period", 0) == 1
    do_classify_flowering = entry.get("classify_flowering", 0) == 1

    safe_species = make_safe_slug(species)

    # --------------------------------------------
    # Standard seasonality and/or flight-period analysis
    # --------------------------------------------
    if classify_seasonality or do_classify_flowering or do_classify_flight_period:
        df = load_data(parameters["start"], species, parameters["location"], False)
        if df is not None:
            monthly_totals, totals_smooth_x, totals_smooth_y = calculate_monthly_totals(df)
            presence, presence_smooth_x, presence_smooth_y = calculate_presence(df)

            # Standard seasonality analysis
            if classify_seasonality:
                filestem = f"year_in_the_life_{safe_species}_{safe_location}"

                export_dict = {
                    "Monthly Totals": monthly_totals,
                    "Presence": presence
                }

                features, classification, reason = classify_species_from_monthly_data(monthly_totals, presence)
                features["species"] = species
                features["location"] = parameters["location"]
                features["classification"] = classification
                features["reason"] = reason

                export_dict["Features"] = pd.DataFrame([features]).T

                classification_summary = pd.concat([
                    classification_summary,
                    pd.DataFrame({
                        "Species": [species],
                        "Classification": [classification],
                        "Reason": [reason]
                    })
                ], ignore_index=True)

                export_to_spreadsheet(export_folder_path, f"{filestem}.xlsx", export_dict)
                chart_monthly_totals(species, filestem, monthly_totals, totals_smooth_x, totals_smooth_y)
                chart_presence(species, filestem, presence, presence_smooth_x, presence_smooth_y)

            # Butterfly flight-period analysis
            if do_classify_flight_period:
                flight_filestem = f"year_in_the_life_{safe_species}_{safe_location}_butterfly_flight_period"

                flight_export_dict = {
                    "Monthly Totals": monthly_totals,
                    "Presence": presence
                }

                flight_features, flight_classification, flight_reason = classify_flight_period_from_monthly_data(
                    monthly_totals,
                    presence
                )

                flight_features["species"] = species
                flight_features["location"] = parameters["location"]
                flight_features["flight_period_classification"] = flight_classification
                flight_features["reason"] = flight_reason

                flight_export_dict["Features"] = pd.DataFrame([flight_features]).T

                flight_period_summary = pd.concat([
                    flight_period_summary,
                    pd.DataFrame({
                        "Species": [species],
                        "Flight Period Classification": [flight_classification],
                        "Reason": [flight_reason]
                    })
                ], ignore_index=True)

                export_to_spreadsheet(export_folder_path, f"{flight_filestem}.xlsx", flight_export_dict)
                chart_monthly_totals(species, flight_filestem, monthly_totals, totals_smooth_x, totals_smooth_y)
                chart_presence(species, flight_filestem, presence, presence_smooth_x, presence_smooth_y)
    
            # Flowering-period analysis
            if do_classify_flowering:
                flowering_filestem = f"year_in_the_life_{safe_species}_{safe_location}"

                flowering_export_dict = {
                    "Monthly Totals": monthly_totals,
                    "Presence": presence
                }

                flowering_features, flowering_classification, flowering_reason = classify_flowering_from_monthly_data(
                    monthly_totals,
                    presence
                )

                flowering_features["species"] = species
                flowering_features["location"] = parameters["location"]
                flowering_features["flowering_classification"] = flowering_classification
                flowering_features["reason"] = flowering_reason

                flowering_export_dict["Features"] = pd.DataFrame([flowering_features]).T

                flowering_summary = pd.concat([
                    flowering_summary,
                    pd.DataFrame({
                        "Species": [species],
                        "Flowering Classification": [flowering_classification],
                        "Reason": [flowering_reason]
                    })
                ], ignore_index=True)

                export_to_spreadsheet(export_folder_path, f"{flowering_filestem}.xlsx", flowering_export_dict)
                chart_monthly_totals(species, flowering_filestem, monthly_totals, totals_smooth_x, totals_smooth_y)
                chart_presence(species, flowering_filestem, presence, presence_smooth_x, presence_smooth_y)

    # ----------------------------
    # Breeding analysis
    # ----------------------------
    if classify_breeding:
        df_breeding = load_data(parameters["start"], species, parameters["location"], True)
        if df_breeding is not None:
            breeding_monthly_totals, breeding_totals_smooth_x, breeding_totals_smooth_y = calculate_monthly_totals(df_breeding)
            breeding_presence, breeding_presence_smooth_x, breeding_presence_smooth_y = calculate_presence(df_breeding)

            breeding_filestem = f"year_in_the_life_{safe_species}_{safe_location}_breeding"

            breeding_export_dict = {
                "Monthly Totals": breeding_monthly_totals,
                "Presence": breeding_presence
            }

            breeding_features, breeding_classification, breeding_reason = classify_breeding_from_monthly_data(
                breeding_monthly_totals,
                breeding_presence
            )

            breeding_features["species"] = species
            breeding_features["location"] = parameters["location"]
            breeding_features["breeding_classification"] = breeding_classification
            breeding_features["reason"] = breeding_reason

            breeding_export_dict["Features"] = pd.DataFrame([breeding_features]).T

            breeding_summary = pd.concat([
                breeding_summary,
                pd.DataFrame({
                    "Species": [species],
                    "Breeding Classification": [breeding_classification],
                    "Reason": [breeding_reason]
                })
            ], ignore_index=True)

            export_to_spreadsheet(export_folder_path, f"{breeding_filestem}.xlsx", breeding_export_dict)
            chart_monthly_totals(
                species,
                breeding_filestem,
                breeding_monthly_totals,
                breeding_totals_smooth_x,
                breeding_totals_smooth_y
            )
            chart_presence(
                species,
                breeding_filestem,
                breeding_presence,
                breeding_presence_smooth_x,
                breeding_presence_smooth_y
            )

# Export the seasonality summary
if len(classification_summary) > 0:
    export_to_spreadsheet(
        export_folder_path,
        f"species_classifications_{safe_location}.xlsx",
        {"Classifications": classification_summary}
    )

# Export the breeding summary
if len(breeding_summary) > 0:
    export_to_spreadsheet(
        export_folder_path,
        f"species_breeding_classifications_{safe_location}.xlsx",
        {"Breeding Classifications": breeding_summary}
    )

# Export the butterfly flight-period summary
if len(flight_period_summary) > 0:
    export_to_spreadsheet(
        export_folder_path,
        f"species_flight_period_classifications_{safe_location}.xlsx",
        {"Flight Period Classifications": flight_period_summary}
    )


# Export the flowering summary
if len(flowering_summary) > 0:
    export_to_spreadsheet(
        export_folder_path,
        f"species_flowering_classifications_{safe_location}.xlsx",
        {"Flowering Classifications": flowering_summary}
    )
